# Generate synthetic data for the agnews dataset
## Llama-2-7b-chat-hf

<https://huggingface.co/meta-llama/Llama-2-7b-hf>  
<https://huggingface.co/meta-llama/Llama-2-7b-chat-hf>

1. baseline
2. targeted + linguistic tags
3. unsupervised context
4. (unsupervised context + linguistic tags)

## Notes
### About chat template

the base *meta-llama/Llama-2-7b-hf* doesn't work quite right for the task of data generation. So I'm going to use the chat version.

After different tests I have seen that the chat version doesn't directly generate the text, but is going to ask for which label to you want it. So I'm going to randomly select a label each time and use the following format:

```python
<s>[INST] <<SYS>>
{{ system_prompt }}
<</SYS>>

{{ user_msg_1 }} [/INST] {{ model_answer_1 }} </s><s>[INST] {{ user_msg_2 }} [/INST]
```

with:
- model_answer_1 = "Of course! I'm happy to help. Please provide me with the category you would like me to focus on, and I will generate a high-quality short document for you."
- user_msg_2 = "label: " + random_label

The actual format with the correct special tokens is going to be generated by the `tokenizer.chat_template()`, we supply a list of messages of the kind:

```python
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_msg_1},
    {"role": "assistant", "content": model_answer_1},
    {"role": "user", "content": user_msg_2},
]
```

#### user_msg_2
- "The label is up to you.": the results tend to be very skewed towards a particular label.
- "label: " + random_label (the used one): works fine, but the cons is that we will have a uniform distribution. And that for the case of the unsupervised context we are "limiting" the options given by the unsupervised context. Instead of "picking" the best context example to look at, this will force a more strinct generation.

### About context examples

the context examples are inserted in the prompt where there is the placeholder `"ADD_CONTEXT_HERE"`, formatted as a bulled list.

There is a similar placeholder for inserting the random label: `"ADD_LABEL_HERE"`

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM, BitsAndBytesConfig

# CHANGE WORKING DIRECTORY TO ROOT
current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..")
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass
else:
    os.chdir("../..")
from src._utils._generate_dataset import main_generate_dataset
from src._utils._helpers import get_generated_examples_df, clear_cuda_cache, get_context_examples

# get true labels
DATASET_NAME = "agnews"
df_real = pd.read_csv("real_data/train/"+DATASET_NAME+"trainAll.csv").rename(
    columns={"2": "text", "3": "label"}
)
correct_labels = df_real["label"].unique().tolist()
labels_str = ", ".join(correct_labels)
labels_str_bullet = "\n".join([f"- {name}" for name in correct_labels])
model = None
HF_TOKEN = open("src/_utils/hf_token.txt","r").read() # your huggingface token

In [2]:
# chat template format:
# <s>[INST] <<SYS>>
# {{ system_prompt }}
# <</SYS>>
# 
# {{ user_msg_1 }} [/INST] {{ model_answer_1 }} </s><s>[INST] {{ user_msg_2 }} [/INST]

PROMPTS = {}
# same for all prompts
system = "You are an expert in journalism and NLP specializing in news classification."
assistant_response = "Of course! I'm happy to help. Please provide me with the category you would like me to focus on, and I will generate a high-quality short document for you."
user_response = "label: ADD_LABEL_HERE" # <- special token

####### BASELINE #######
prompt_baseline = f"""\
Your task is to generate one high-quality short document (around 30 words), that talks about one of the following four News categories:  
{labels_str_bullet}

Choose one of the categories (labels), generate the corresponding text and return it in the following JSON format:

```json
{{
    "text": "<text of the document>", 
    "label": "<corresponding label>", 
}}
```
"""
PROMPTS["baseline"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_baseline},
    {"role": "assistant", "content": assistant_response},
    {"role": "user", "content": user_response},
]


####### TARGETED #######
prompt_targeted =  f"""\
Your task is to generate one high-quality short document (around 30 words), that talks about one of the following four News categories:  
{labels_str_bullet}

For each example, also list the key phenomena it covers.

### **Follow these topics:**
- **Business**  
  - Markets  
  - Economy  
  - Companies  
  - Startups  
  - Regulations  

- **Sci/Tech**  
  - AI  
  - Space  
  - Cybersecurity  
  - Biotech  
  - Climate  

- **Sports**  
  - Events  
  - Records  
  - Highlights  
  - Scandals  
  - Olympics  

- **World**  
  - Politics  
  - Conflicts  
  - Disasters  
  - Human Rights  
  - Trade

### **Output Format (JSON)**
The labels must be one of the specified categories, which are: {labels_str}. \
Choose one of the categories (labels), generate the corresponding text, write the corresponding phenomena and return it in the following JSON format:

```json
{{
    "text": "<text of the document>", 
    "label": "<corresponding label>", 
    "phenomena": ["<phenomenon1>", "<phenomenon2>", ...]
}}
```
"""
PROMPTS["targeted + linguistic tags"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_targeted},
    {"role": "assistant", "content": assistant_response},
    {"role": "user", "content": user_response},
]


####### UNSUPERVISED CONTEXT #######
# 'ADD_CONTEXT_HERE' is a placeholder for the context that will be added at each iteration
prompt_unsupervised = f"""\
Your task is to generate an high-quality short documents (around 30 words), that talks about one of the following four News categories (labels):
{labels_str_bullet}

Here some examples of the documents you can use as a reference:
ADD_CONTEXT_HERE

Choose one of the categories (labels), generate the corresponding text and return it in the following JSON format:

```json
[
    {{
        "text": "<text of the document>", 
        "label": "<corresponding label>",
    }}
]
```
"""
PROMPTS["unsupervised context"] = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt_unsupervised},
    {"role": "assistant", "content": assistant_response},
    {"role": "user", "content": user_response},
]

# Llama-2-7b-chat-hf

In [3]:
#############################################
# LOAD MODEL
#############################################

if model:
    clear_cuda_cache(model)

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model_name = "meta-llama/Llama-2-7b-chat-hf"
model = LlamaForCausalLM.from_pretrained(
            model_name, 
            token=HF_TOKEN,
            torch_dtype=torch.float16,
            attn_implementation='flash_attention_2',
            quantization_config=quantization_config,
            low_cpu_mem_usage=True
        ).to("cuda")

tokenizer = LlamaTokenizer.from_pretrained(model_name, token=HF_TOKEN)

OUTPUT_DIR = "synthetic_data/datasets/Llama-2-7b-chat-hf/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
base_config = {
    "dataset": DATASET_NAME,
    "model": model,
    "tokenizer": tokenizer,
    # "generation_method": "baseline",
    #### We are going to give as input the list of messages
    # "prompt": {system:..., user: prompt, assistant: None},
    "apply_chat_template": True,
    "num_examples": 500,
    "max_new_tokens": 1024,
    "seed": 42,
    #"json_output_file": OUTPUT_DIR+"agnews_baseline_500.json",
    "log_file": OUTPUT_DIR+"generate_dataset_"+DATASET_NAME+"_log.json",
    "correct_labels": correct_labels,
    "correct_fields": ["text", "label"],
    ### context
    # "context_examples": None,
    # "prompt_postfix": None,
    "verbose": False,
    ### At each iteration we ask randomly for a label
    "add_random_label": True, 
}

### 1. baseline

In [5]:
name = "baseline"
config = base_config.copy()
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+DATASET_NAME+"_baseline_500.json"
main_generate_dataset(config)

Generating Examples:  35%|███▍      | 173/500 [05:55<12:58,  2.38s/ex, examples=173/500, run=175]

❌ Invalid example at run 175


Generating Examples:  65%|██████▌   | 325/500 [11:07<06:09,  2.11s/ex, examples=325/500, run=328]

❌ Invalid example at run 328


Generating Examples:  79%|███████▉  | 397/500 [13:40<03:53,  2.26s/ex, examples=397/500, run=401]

❌ Invalid example at run 401


Generating Examples: 100%|██████████| 500/500 [17:14<00:00,  2.07s/ex, examples=500/500, run=504]

⏱️ Time taken: 1034.71 seconds.


### 2. targeted + linguistic tags

In [6]:
config = base_config.copy()
name = "targeted + linguistic tags"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+DATASET_NAME+"_targeted+tags_500.json"
config["correct_fields"] = ["text", "label", "phenomena"]
main_generate_dataset(config)

Generating Examples:   3%|▎         | 16/500 [00:50<24:49,  3.08s/ex, examples=16/500, run=17]

❌ Failed to parse generation 18: Expecting ',' delimiter: line 3 column 5 (char 468)


Generating Examples:   4%|▍         | 21/500 [01:09<25:11,  3.16s/ex, examples=21/500, run=23]

❌ Failed to parse generation 24: Expecting ',' delimiter: line 3 column 5 (char 444)


Generating Examples:   5%|▌         | 25/500 [01:26<24:39,  3.12s/ex, examples=25/500, run=28]

...

❌ Failed to parse generation 504: Expecting ',' delimiter: line 3 column 5 (char 557)


Generating Examples:  98%|█████████▊| 492/500 [26:40<00:35,  4.45s/ex, examples=492/500, run=506]

❌ Failed to parse generation 507: Expecting ',' delimiter: line 3 column 5 (char 433)


Generating Examples: 100%|██████████| 500/500 [27:07<00:00,  3.26s/ex, examples=500/500, run=515]

⏱️ Time taken: 1627.52 seconds.


In [7]:
config = base_config.copy()
name = "targeted + linguistic tags"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+DATASET_NAME+"_targeted+tags_500.json"
config["correct_fields"] = ["text", "label", "phenomena"]

In [8]:
### Generate other 500 examples

config['seed'] = (config['seed'] + 1)*8
config['json_output_file'] = OUTPUT_DIR+DATASET_NAME+"_targeted+tags_500_2.json"
main_generate_dataset(config)

Generating Examples:  18%|█▊        | 88/500 [04:39<21:08,  3.08s/ex, examples=88/500, run=89]

❌ Failed to parse generation 90: Expecting ',' delimiter: line 3 column 5 (char 526)


Generating Examples:  19%|█▊        | 93/500 [04:56<19:23,  2.86s/ex, examples=93/500, run=95]

❌ Failed to parse generation 96: Expecting ',' delimiter: line 3 column 5 (char 634)


Generating Examples:  23%|██▎       | 117/500 [06:09<15:39,  2.45s/ex, examples=117/500, run=120]

...

❌ Failed to parse generation 413: Expecting ',' delimiter: line 3 column 5 (char 762)


Generating Examples:  99%|█████████▉| 494/500 [26:11<00:21,  3.51s/ex, examples=494/500, run=502]

❌ Failed to parse generation 503: Expecting ',' delimiter: line 3 column 5 (char 563)


Generating Examples: 100%|██████████| 500/500 [26:27<00:00,  3.18s/ex, examples=500/500, run=509]

⏱️ Time taken: 1587.8 seconds.


### 3. unsupervised context

In each prompt we attach n (5) examples sampled randomly from the train set. The samples are used without the labels, so they works as unsupervised context for the model, when we will generate the new synthetic sample. 

In [9]:
# we take more than 500 because it can happen that some prompt
# generate the example in the wrong format, so is not read correctly (and discarded)
num_prompts = 1000
num_examples_per_prompt = 5
np.random.seed(42)
context_examples = get_context_examples(df_real, num_examples_per_prompt, num_prompts)
print(f"Number prompts: {len(context_examples)}")
print(f"Number of examples per prompt: {len(context_examples[0])}")
print(context_examples[0])

Number prompts: 1000
Number of examples per prompt: 5
['With Derek Lowe #39;s days in Boston almost certainly numbered -- and Pedro Martinez #39;s future here in question -- the Red Sox are expected to greet All-Star righthander Carl Pavano this week on Yawkey Way.', 'Symantec has released firmware fixes for a string of critical security holes in its firewall/VPN and Gateway Security products which could be exploited to cause a denial of service, identify ', 'King County prosecutors charged a Covington orthodontist yesterday with engaging in sexually explicit Internet conversations with several girls, including three current or former patients, and with dealing child pornography.', 'LONDON (Reuters) - Oil producers are emerging to lock in record high prices for their future crude output, but activity is modest as firms still fear calling a premature end to this year #39;s stunning price rise, traders said on Friday. ', 'STOCKHOLM (AFP) - Andre Agassi was set to intensify his chase for 

In [10]:
config = base_config.copy()
name = "unsupervised context"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+DATASET_NAME+"_unsupervisedContext_500.json"
config["context_examples"] = context_examples
main_generate_dataset(config)

Generating Examples (context):  12%|█▏        | 62/500 [02:18<15:02,  2.06s/ex, examples=62/500, run=62]

❌ Invalid example at run 62


Generating Examples (context):  13%|█▎        | 66/500 [02:28<16:06,  2.23s/ex, examples=66/500, run=66]

❌ Failed to parse generation 67: Expecting ',' delimiter: line 3 column 1 (char 523)


Generating Examples (context):  14%|█▍        | 70/500 [02:38<16:39,  2.32s/ex, examples=70/500, run=72]

...

❌ Invalid example at run 509


Generating Examples (context): 100%|█████████▉| 499/500 [19:12<00:02,  2.13s/ex, examples=499/500, run=540]

❌ Failed to parse generation 541: Expecting ',' delimiter: line 3 column 1 (char 321)


Generating Examples (context): 100%|██████████| 500/500 [19:14<00:00,  2.31s/ex, examples=500/500, run=542]

⏱️ Time taken: 1154.35 seconds.


In [11]:
generated_df, _ = get_generated_examples_df(OUTPUT_DIR+DATASET_NAME+"_unsupervisedContext_500.json")

for i in range(5):
    print("TEXT: "+generated_df.iloc[i]['text'])
    print("LABEL: "+generated_df.iloc[i]['label'])
    print("CONTEXT EXAMPLES:")
    for j in range(len(generated_df.iloc[i]['context_examples'])):
        print("- "+generated_df.iloc[i]['context_examples'][j])

    print("\n"+"=="*50)

TEXT: Lionel Messi scores hat-trick as Barcelona beats Real Madrid 3-2 in El Clásico
LABEL: Sports
CONTEXT EXAMPLES:
- With Derek Lowe #39;s days in Boston almost certainly numbered -- and Pedro Martinez #39;s future here in question -- the Red Sox are expected to greet All-Star righthander Carl Pavano this week on Yawkey Way.
- Symantec has released firmware fixes for a string of critical security holes in its firewall/VPN and Gateway Security products which could be exploited to cause a denial of service, identify 
- King County prosecutors charged a Covington orthodontist yesterday with engaging in sexually explicit Internet conversations with several girls, including three current or former patients, and with dealing child pornography.
- LONDON (Reuters) - Oil producers are emerging to lock in record high prices for their future crude output, but activity is modest as firms still fear calling a premature end to this year #39;s stunning price rise, traders said on Friday. 
- STOCKHO

In [12]:
# Unsupervised context + tags
# ...